# Topic Modelling LDA — TFG UAM

## Instalación

In [ ]:
!pip install gensim pyLDAvis spacy pandas matplotlib seaborn ipywidgets -q
!python -m spacy download es_core_news_sm -q
print("Instalación completada")

## Imports, carga y preprocesamiento

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import unicodedata
import spacy
from collections import Counter
from tqdm import tqdm

import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel, CoherenceModel
from gensim.models.phrases import Phrases
import pyLDAvis
import pyLDAvis.gensim_models

plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
})

AZUL    = "#2E5496"
ROJO    = "#C0392B"
VERDE   = "#27AE60"
NARANJA = "#E67E22"
PALETA  = [AZUL, ROJO, VERDE, NARANJA, "#8E44AD", "#16A085",
           "#D35400", "#2C3E50", "#1ABC9C", "#E74C3C",
           "#3498DB", "#F39C12", "#27AE60", "#E91E63", "#795548"]
ORDEN_SENT = ["MUY_POSITIVO","POSITIVO","NEUTRO","NEGATIVO","MUY_NEGATIVO"]
COLORES_SA = {
    "MUY_POSITIVO": "#1a5276", "POSITIVO": "#2980b9",
    "NEUTRO": "#95a5a6", "NEGATIVO": "#e74c3c", "MUY_NEGATIVO": "#922b21",
}
SCORE_MAP = {"MUY_POSITIVO": 2, "POSITIVO": 1, "NEUTRO": 0,
             "NEGATIVO": -1, "MUY_NEGATIVO": -2}

# ── Rutas ─────────────────────────────────────────────────────────────────────
# Ajusta CARPETA si tus archivos están en otro directorio
CARPETA = "."
# Ejemplo ruta absoluta Windows:
# CARPETA = r"C:/Users/feliz/Documents/TFG"
# Ejemplo Google Colab con Drive:
# from google.colab import drive; drive.mount("/content/drive")
# CARPETA = "/content/drive/MyDrive/Colab Notebooks/sentimental analisis"

df_tw = pd.read_csv(f"{CARPETA}/corpus_tweets_sentiment.csv")
df_rv = pd.read_csv(f"{CARPETA}/resenas_sentiment.csv")
df_tw["sentiment_score_num"] = df_tw["sentiment_label"].map(SCORE_MAP)
df_rv["sentiment_score_num"] = df_rv["sentiment_label"].map(SCORE_MAP)
print(f"Tweets: {len(df_tw):,}  |  Reseñas: {len(df_rv):,}")

# ── Stopwords LDA ─────────────────────────────────────────────────────────────
STOPWORDS_LDA = {
    "de","la","el","en","que","y","a","los","las","por","con","se","una","un",
    "es","al","del","lo","su","para","como","mas","pero","sus","le","ya","o",
    "este","ha","me","hay","sobre","fue","ser","son","muy","no","mi","te","han",
    "he","tu","yo","nos","les","has","era","ni","cuando","porque","donde","asi",
    "entre","hasta","desde","tras","cada","todo","esta","si","tan","solo","bien",
    "todos","todas","aqui","ahi","hoy","vez","sido","tiene","parte","otros",
    "menos","otro","mismo","dicho","nuestro","nuestra","ellos","ellas","cual",
    "ante","bajo","gran","algo","poder","hacer","decir","tener","estar","haber",
    "uam","madrid","autonoma","universidad","uam_madrid","cantoblanco",
    "hacer","hecho","tengo","puede","quiero","quiere","seria","siendo",
    "vamos","viene","trata","lleva","sigue","poner","pone","pasa","parece",
    "creo","cree","nuevo","buena","bueno","primer","primera","dado","dar",
    "https","http","rt","via","ano","anos","mes","dia","dias","semana",
    "manana","ahora","antes","despues","siempre","nunca","tarde","pronto",
    "bien","mal","mejor","peor","mucho","poco","nada","todo",
    "gracias","enhorabuena","felicidades","suerte","favor",
}

# ── Lematización ──────────────────────────────────────────────────────────────
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])

def lematizar_texto(texto):
    t = str(texto).lower()
    t = "".join(c for c in unicodedata.normalize("NFD", t)
                if unicodedata.category(c) != "Mn")
    t = re.sub(r"https?://\S+", "", t)
    t = re.sub(r"@\w+", "", t)
    t = re.sub(r"#(\w+)", r"\1", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    doc = nlp(t[:1000])
    return [
        token.lemma_ for token in doc
        if token.pos_ in ("NOUN", "VERB", "ADJ")
        and len(token.lemma_) >= 3
        and token.lemma_ not in STOPWORDS_LDA
        and not token.is_stop
        and token.is_alpha
    ]

# Test
print("Test:", lematizar_texto("Los profesores son excelentes aunque la secretaría es un desastre"))

print("\nLematizando tweets...")
docs_tw = [lematizar_texto(t) for t in tqdm(df_tw["text_clean"].fillna("").tolist())]

print("Lematizando reseñas...")
docs_rv = [lematizar_texto(t) for t in tqdm(df_rv["text_clean"].fillna("").tolist())]

# Bigrams
print("\nDetectando bigrams...")
todos = docs_tw + docs_rv
bigram_model = Phrases(todos, min_count=10, threshold=8)
docs_tw_bg = [bigram_model[d] for d in docs_tw]
docs_rv_bg = [bigram_model[d] for d in docs_rv]

bgs = [t for d in docs_tw_bg for t in d if "_" in t]
print("Top bigrams:", Counter(bgs).most_common(10))

# Corpus LDA
def hacer_corpus(docs, no_below=5, no_above=0.70):
    d = corpora.Dictionary(docs)
    d.filter_extremes(no_below=no_below, no_above=no_above)
    c = [d.doc2bow(doc) for doc in docs]
    print(f"  Vocabulario: {len(d)} términos | Documentos: {len(c)}")
    return d, c

print("\nCorpus tweets:")
dict_tw, corpus_tw = hacer_corpus(docs_tw_bg)
print("Corpus reseñas:")
dict_rv, corpus_rv = hacer_corpus(docs_rv_bg, no_below=3)
print("\n✅ Preprocesamiento completado")


## Optimización del número de temas

In [ ]:
def optimizar_k(corpus, dictionary, docs, k_min=5, k_max=15, seed=42):
    resultados = []
    for k in tqdm(range(k_min, k_max + 1)):
        modelo = LdaModel(
            corpus=corpus, id2word=dictionary, num_topics=k,
            random_state=seed, passes=15, iterations=100,
            alpha="auto", eta="auto",
        )
        coh = CoherenceModel(model=modelo, texts=docs,
                             dictionary=dictionary, coherence="c_v").get_coherence()
        resultados.append((k, round(coh, 4), modelo))
        print(f"  k={k:2d} → coherencia={coh:.4f}")
    return resultados

print("Optimizando tweets (k=5..15)...")
res_tw = optimizar_k(corpus_tw, dict_tw, docs_tw_bg)

print("\nOptimizando reseñas (k=3..10)...")
res_rv = optimizar_k(corpus_rv, dict_rv, docs_rv_bg, k_min=3, k_max=10)

# Selección automática
cohs_tw = [r[1] for r in res_tw]; cohs_rv = [r[1] for r in res_rv]
k_opt_tw = res_tw[cohs_tw.index(max(cohs_tw))][0]
k_opt_rv = res_rv[cohs_rv.index(max(cohs_rv))][0]
lda_tw   = res_tw[cohs_tw.index(max(cohs_tw))][2]
lda_rv   = res_rv[cohs_rv.index(max(cohs_rv))][2]
print(f"\n k óptimo tweets:  {k_opt_tw}")
print(f" k óptimo reseñas: {k_opt_rv}")

# ── Fig 13 ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor="white")
for ax, res, titulo, k_opt in zip(
        axes,
        [res_tw, res_rv],
        [f"Tweets (rango k=5–15)", f"Reseñas Google (rango k=3–10)"],
        [k_opt_tw, k_opt_rv]):
    ax.set_facecolor("white")
    ks = [r[0] for r in res]; cohs = [r[1] for r in res]
    ax.plot(ks, cohs, color=AZUL, linewidth=2.5, marker="o",
            markersize=8, markerfacecolor="white", markeredgewidth=2,
            label="Coherencia c_v", zorder=3)
    ax.axvline(k_opt, color=ROJO, linewidth=1.8, linestyle="--",
               label=f"k óptimo = {k_opt}", zorder=2)
    ax.scatter([k_opt], [max(cohs)], color=ROJO, s=120, zorder=5)
    # Etiqueta valor en cada punto
    for k, c in zip(ks, cohs):
        ax.annotate(f"{c:.3f}", (k, c), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=7.5, color="#444444")
    ax.set_xlabel("Número de temas (k)", color="#222222", fontsize=11)
    ax.set_ylabel("Coherencia c_v", color="#222222", fontsize=11)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color="#222222", pad=8)
    ax.set_xticks(ks)
    ax.tick_params(colors="#222222")
    ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=10, frameon=True, facecolor="white",
              edgecolor="#CCCCCC", labelcolor="#222222")

fig.suptitle("Fig. 13 — Optimización del número de temas LDA\n"
             "(se selecciona el k con mayor coherencia c_v)",
             fontsize=13, fontweight="bold", color="#222222", y=1.03)
plt.tight_layout()
plt.savefig(f"{CARPETA}/fig13_optimizacion_k.png",
            bbox_inches="tight", dpi=150, facecolor="white")
plt.show()


## Palabras clave de cada tema

In [ ]:
def mostrar_temas(modelo, titulo):
    print(f"\n{'='*60}\nTEMAS — {titulo}\n{'='*60}")
    for i in range(modelo.num_topics):
        pals = " | ".join([f"{p}({w:.3f})" for p, w in modelo.show_topic(i, topn=12)])
        print(f"  Tema {i+1}: {pals}")

mostrar_temas(lda_tw, f"TWEETS (k={k_opt_tw})")
mostrar_temas(lda_rv, f"RESEÑAS (k={k_opt_rv})")

def grafico_palabras(modelo, titulo_fig, archivo, n_words=10):
    k = modelo.num_topics
    ncols = min(3, k)
    nrows = (k + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(7*ncols, 5*nrows), facecolor="white")
    axes = np.array(axes).flatten() if k > 1 else [axes]

    for i in range(k):
        ax = axes[i]; ax.set_facecolor("white")
        pals = modelo.show_topic(i, topn=n_words)
        ws = [w for _, w in pals]; ps = [p for p, _ in pals]
        color = PALETA[i % len(PALETA)]
        bars = ax.barh(range(n_words), ws[::-1], color=color,
                       alpha=0.85, zorder=3, edgecolor="white")
        ax.set_yticks(range(n_words))
        ax.set_yticklabels(ps[::-1], fontsize=11, color="#111111")
        ax.set_xlabel("Peso en el tema", color="#222222", fontsize=10)
        ax.set_title(f"Tema {i+1}", fontsize=12, fontweight="bold",
                     color="#222222", pad=6)
        ax.tick_params(colors="#222222")
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
        ax.grid(axis="x", alpha=0.2)
        # Valores al final de cada barra
        for bar, val in zip(bars, ws[::-1]):
            ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                    f"{val:.3f}", va="center", fontsize=8, color="#444444")

    for j in range(k, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"Fig. 14 — Palabras clave por tema LDA\n{titulo_fig}",
                 fontsize=13, fontweight="bold", color="#222222", y=1.01)
    plt.tight_layout(pad=2)
    plt.savefig(f"{CARPETA}/{archivo}", bbox_inches="tight",
                dpi=150, facecolor="white")
    plt.show()

grafico_palabras(lda_tw, f"Tweets (k={k_opt_tw})", "fig14a_temas_tweets.png")
grafico_palabras(lda_rv, f"Reseñas Google (k={k_opt_rv})", "fig14b_temas_resenas.png")


## Etiquetado manual



In [ ]:

NOMBRES_TEMAS_TW = {
    0:  "T1 — Comunidad universitaria y salud",
    1:  "T2 — Campus y vida universitaria",
    2:  "T3 — Convocatorias y movilidad",
    3:  "T4 — Calidad docente y educativa",
    4:  "T5 — Máster y formación profesional",
    5:  "T6 — Salud, ciencia y sociedad",
    6:  "T7 — Experiencia académica estudiantil",
    7:  "T8 — Exámenes y acceso universitario",
    8:  "T9 — Valoración y experiencia general",
    9:  "T10 — Debate público y controversia",
    10: "T11 — Trayectorias y futuro profesional",
    11: "T12 — Gestión y convocatorias internas",
    12: "T13 — Investigación y vida académica",
}

NOMBRES_TEMAS_RV = {
    0: "T1 — Entorno físico y campus",
    1: "T2 — Valoración global de calidad",
    2: "T3 — Experiencia académica y personal",
    3: "T4 — Oferta académica y recomendación",
}

## Visualización interactiva pyLDAvis

In [ ]:
pyLDAvis.enable_notebook()

print("Preparando visualización tweets...")
vis_tw = pyLDAvis.gensim_models.prepare(lda_tw, corpus_tw, dict_tw, sort_topics=False)
pyLDAvis.save_html(vis_tw, f"{CARPETA}/lda_tweets_interactive.html")
print("Guardado lda_tweets_interactive.html")
vis_tw


In [ ]:
print("Preparando visualización reseñas...")
vis_rv = pyLDAvis.gensim_models.prepare(lda_rv, corpus_rv, dict_rv, sort_topics=False)
pyLDAvis.save_html(vis_rv, f"{CARPETA}/lda_resenas_interactive.html")
print("✅ Guardado lda_resenas_interactive.html")
vis_rv


## Asignación de tema dominante + ABSA

In [ ]:
def asignar_temas(modelo, corpus_bow, nombres):
    res = []
    for doc in corpus_bow:
        dist = modelo.get_document_topics(doc, minimum_probability=0)
        tid, prob = max(dist, key=lambda x: x[1]) if dist else (0, 0.0)
        res.append((tid, nombres.get(tid, f"Tema {tid+1}"), round(float(prob), 4)))
    return res

res_tw = asignar_temas(lda_tw, corpus_tw, NOMBRES_TEMAS_TW)
df_tw["topic_id"]     = [r[0] for r in res_tw]
df_tw["topic_nombre"] = [r[1] for r in res_tw]
df_tw["topic_prob"]   = [r[2] for r in res_tw]

res_rv = asignar_temas(lda_rv, corpus_rv, NOMBRES_TEMAS_RV)
df_rv["topic_id"]     = [r[0] for r in res_rv]
df_rv["topic_nombre"] = [r[1] for r in res_rv]
df_rv["topic_prob"]   = [r[2] for r in res_rv]

print("Distribución temas tweets:\n", df_tw["topic_nombre"].value_counts())
print("\nDistribución temas reseñas:\n", df_rv["topic_nombre"].value_counts())

def grafico_absa(df, titulo, archivo, nombres):
    temas = [nombres[k] for k in sorted(nombres.keys())]
    # Etiquetas cortas para los ejes (quitar "Tema N — ")
    etiq = [t.split("—")[-1].strip() if "—" in t else t for t in temas]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor="white")

    # ── Panel izquierdo: barras apiladas % ────────────────────────────────────
    ax = axes[0]; ax.set_facecolor("white")
    absa = df.groupby(["topic_nombre","sentiment_label"]).size().unstack(fill_value=0)
    pct  = absa.div(absa.sum(axis=1), axis=0).reindex(index=temas, fill_value=0)
    pct  = pct.reindex(columns=ORDEN_SENT, fill_value=0) * 100
    bottom = np.zeros(len(pct))
    for clase in ORDEN_SENT:
        vals = pct[clase].values if clase in pct.columns else np.zeros(len(pct))
        ax.bar(range(len(pct)), vals, bottom=bottom,
               color=COLORES_SA[clase], label=clase.replace("_"," "),
               width=0.6, edgecolor="white", linewidth=0.5)
        bottom += vals
    ax.set_xticks(range(len(temas)))
    ax.set_xticklabels(etiq, rotation=35, ha="right", fontsize=10, color="#111111")
    ax.set_ylabel("% de documentos", color="#222222", fontsize=11)
    ax.set_ylim(0, 105)
    ax.set_title("Distribución de sentimiento por tema (%)",
                 fontsize=11, fontweight="bold", color="#222222", pad=8)
    ax.tick_params(colors="#222222")
    ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
    ax.grid(axis="y", alpha=0.2)
    # Leyenda fuera del área de barras
    ax.legend(fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.22),
              ncol=5, frameon=True, facecolor="white",
              edgecolor="#CCCCCC", labelcolor="#222222")

    # ── Panel derecho: score medio con IC ─────────────────────────────────────
    ax2 = axes[1]; ax2.set_facecolor("white")
    sc = df.groupby("topic_nombre")["sentiment_score_num"].agg(["mean","sem","count"]).reindex(temas)
    sc["ic"] = sc["sem"] * 1.96
    colores = [ROJO if m < 0 else AZUL for m in sc["mean"]]
    bars2 = ax2.bar(range(len(sc)), sc["mean"], yerr=sc["ic"],
                    color=colores, width=0.55, zorder=3, edgecolor="white",
                    capsize=5, error_kw={"color":"#555555","lw":1.5})
    ax2.axhline(0, color="#AAAAAA", linewidth=0.8, linestyle=":")
    ax2.set_xticks(range(len(sc)))
    ax2.set_xticklabels(etiq, rotation=35, ha="right", fontsize=10, color="#111111")
    ax2.set_ylabel("Score medio de sentimiento\n(-2 muy neg. → +2 muy pos.)",
                   color="#222222", fontsize=11)
    ax2.set_title("Score medio por tema (IC 95%)",
                  fontsize=11, fontweight="bold", color="#222222", pad=8)
    for i, (_, row) in enumerate(sc.iterrows()):
        ax2.text(i, row["mean"] + row["ic"] + 0.05,
                 f"{row['mean']:+.2f}\n(n={int(row['count'])})",
                 ha="center", va="bottom", fontsize=9,
                 fontweight="bold", color="#222222")
    ax2.tick_params(colors="#222222")
    ax2.spines["left"].set_color("#DDDDDD"); ax2.spines["bottom"].set_color("#DDDDDD")
    ax2.grid(axis="y", alpha=0.2)

    fig.suptitle(f"Fig. 15 — ABSA: sentimiento por tema — {titulo}",
                 fontsize=13, fontweight="bold", color="#222222", y=1.01)
    plt.tight_layout(pad=2)
    plt.savefig(f"{CARPETA}/{archivo}", bbox_inches="tight",
                dpi=150, facecolor="white")
    plt.show()

    print(f"\nRanking temas por score ({titulo}):")
    for tema, row in sc.sort_values("mean", ascending=False).iterrows():
        n = int(row["count"])
        print(f"  {row['mean']:+.3f} | {tema} (n={n})")

grafico_absa(df_tw, "Tweets",         "fig15a_absa_tweets.png",  NOMBRES_TEMAS_TW)
grafico_absa(df_rv, "Reseñas Google", "fig15b_absa_resenas.png", NOMBRES_TEMAS_RV)


## Evolución temporal de los temas

In [ ]:
df_tw["año_mes"] = df_tw["año_mes"].astype(str)
evol = df_tw.groupby(["año_mes","topic_nombre"]).size().unstack(fill_value=0)
evol_pct = evol.div(evol.sum(axis=1), axis=0) * 100
meses = list(evol_pct.index)
temas = list(evol_pct.columns)

fig, ax = plt.subplots(figsize=(16, 6), facecolor="white")
ax.set_facecolor("white")

for i, tema in enumerate(temas):
    mm = pd.Series(evol_pct[tema].values).rolling(3, center=True).mean()
    etiq_tema = tema.split("—")[-1].strip() if "—" in tema else tema
    ax.plot(range(len(meses)), mm, color=PALETA[i % len(PALETA)],
            linewidth=2.2, label=etiq_tema, zorder=3)

# Eje X — etiquetas cada 3 meses
etiq_x = []
for m in meses:
    año, mes_n = m.split("-")
    if mes_n in ("01","04","07","10"):
        etiq_x.append(f"{mes_n}/{año[2:]}")
    else:
        etiq_x.append("")

ax.set_xticks(range(len(meses)))
ax.set_xticklabels(etiq_x, rotation=45, ha="right", fontsize=9, color="#222222")
ax.set_xlabel("Mes / Año", color="#222222", fontsize=11)
ax.set_ylabel("% de tweets del mes\n(media móvil 3 meses)", color="#222222", fontsize=11)
ax.set_title("Fig. 16 — Evolución temporal de los temas LDA (tweets, 2023–2026)\n"
             "Cada línea representa el porcentaje mensual de tweets asignados a ese tema",
             fontsize=12, fontweight="bold", color="#222222", pad=10)
ax.tick_params(colors="#222222")
ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
ax.grid(alpha=0.2)

# Leyenda fuera del gráfico, debajo
ax.legend(fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.22),
          ncol=3, frameon=True, facecolor="white",
          edgecolor="#CCCCCC", labelcolor="#222222")

plt.tight_layout(pad=2)
plt.savefig(f"{CARPETA}/fig16_evolucion_temas.png",
            bbox_inches="tight", dpi=150, facecolor="white")
plt.show()


## Ejemplos por tema y exportación

In [ ]:
# Ejemplos representativos
for fuente, df_f, nombres in [("TWEETS", df_tw, NOMBRES_TEMAS_TW),
                               ("RESEÑAS", df_rv, NOMBRES_TEMAS_RV)]:
    print(f"\n{'='*60}\nEJEMPLOS — {fuente}\n{'='*60}")
    for tid, nombre in sorted(nombres.items()):
        sub = df_f[df_f["topic_id"] == tid].nlargest(3, "topic_prob")
        print(f"\n▶ {nombre}")
        for _, row in sub.iterrows():
            texto = row.get("text", row.get("texto",""))
            print(f"  [{row.get('sentiment_label','?')}] {str(texto)[:160]}")

# Exportar
df_tw.to_csv(f"{CARPETA}/corpus_tweets_lda.csv", index=False, encoding="utf-8-sig")
df_rv.to_csv(f"{CARPETA}/resenas_lda.csv",        index=False, encoding="utf-8-sig")
lda_tw.save(f"{CARPETA}/lda_model_tweets.model")
lda_rv.save(f"{CARPETA}/lda_model_resenas.model")

print("\n corpus_tweets_lda.csv")
print(" resenas_lda.csv")
print(" Modelos LDA guardados")
print(" Figuras 13-16 guardadas")
